In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# Dataset
# -----------------------------
sentences = [
    "i love ai",
    "i love python",
    "i hate bugs",
    "bugs are annoying",
    "python is amazing",
    "i hate errors"
]

labels = [1, 1, 0, 0, 1, 0]

# -----------------------------
# Build Vocabulary
# -----------------------------
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for sentence in sentences:
    for word in sentence.split():
        if word not in vocab:
            vocab[word] = len(vocab)

# -----------------------------
# Encode Sentences
# -----------------------------
def encode(sentence):
    return [vocab.get(word, vocab["<UNK>"]) for word in sentence.split()]

encoded_sentences = [encode(sentence) for sentence in sentences]

# -----------------------------
# Padding
# -----------------------------
max_len = max(len(sentence) for sentence in encoded_sentences)

padded_sentences = [
    sentence + [vocab["<PAD>"]] * (max_len - len(sentence))
    for sentence in encoded_sentences
]

# -----------------------------
# Dataset & DataLoader
# -----------------------------
X = torch.tensor(padded_sentences)
y = torch.tensor(labels)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# -----------------------------
# LSTM Model
# -----------------------------
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 12)
        self.lstm = nn.LSTM(12, 20, batch_first=True)
        self.fc = nn.Linear(20, 2)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden[-1])

# -----------------------------
# GRU Model
# -----------------------------
class SentimentGRU(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 12)
        self.gru = nn.GRU(12, 20, batch_first=True)
        self.fc = nn.Linear(20, 2)

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.gru(x)
        return self.fc(hidden[-1])

# -----------------------------
# Training Function
# -----------------------------
def train_model(model):

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(100):

        model.train()

        correct = 0
        total = 0

        for inputs, targets in loader:

            outputs = model(inputs)

            loss = criterion(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            predictions = outputs.argmax(dim=1)

            correct += (predictions == targets).sum().item()
            total += targets.size(0)

    accuracy = 100 * correct / total

    return model, loss.item(), accuracy

# -----------------------------
# Train LSTM
# -----------------------------
lstm_model = SentimentLSTM(len(vocab))

lstm_model, lstm_loss, lstm_acc = train_model(lstm_model)

# -----------------------------
# Train GRU
# -----------------------------
gru_model = SentimentGRU(len(vocab))

gru_model, gru_loss, gru_acc = train_model(gru_model)

# -----------------------------
# Count Parameters
# -----------------------------
lstm_params = sum(p.numel() for p in lstm_model.parameters())

gru_params = sum(p.numel() for p in gru_model.parameters())

# -----------------------------
# Comparison
# -----------------------------
print("-" * 55)
print(f"{'Model':<10}{'Loss':<12}{'Accuracy':<15}{'Parameters'}")
print("-" * 55)
print(f"{'LSTM':<10}{lstm_loss:<12.4f}{lstm_acc:<15.2f}{lstm_params}")
print(f"{'GRU':<10}{gru_loss:<12.4f}{gru_acc:<15.2f}{gru_params}")
print("-" * 55)

-------------------------------------------------------
Model     Loss        Accuracy       Parameters
-------------------------------------------------------
LSTM      0.0001      100.00         2918
GRU       0.0002      100.00         2238
-------------------------------------------------------
